In [1]:
import json
from typing import Any, Dict, List

In [2]:
class JSONComparator:
    def __init__(self):
        """
        Initialize the JSONComparator.
        """
        self.errors = []

    def compare_files(self, base_file: str, target_file: str) -> List[str]:
        """
        Compare two JSON files.
        :param base_file: Path to the base JSON file.
        :param target_file: Path to the target JSON file.
        :return: A list of discrepancies.
        """
        self.errors.clear()
        try:
            with open(base_file, 'r') as bf, open(target_file, 'r') as tf:
                base_json = json.load(bf)
                target_json = json.load(tf)

            self._compare_recursive(base_json, target_json, path="root")
        except json.JSONDecodeError as e:
            self.errors.append(f"Error decoding JSON: {e}")
        except FileNotFoundError as e:
            self.errors.append(f"File not found: {e}")
        except Exception as e:
            self.errors.append(f"Unexpected error: {e}")
        return self.errors

    def _compare_recursive(self, base: Any, comp: Any, path: str):
        """
        Recursive helper to compare two JSON elements dynamically.
        :param base: The base JSON element.
        :param comp: The comparison JSON element.
        :param path: The JSON path for context in errors.
        """
        if type(base) is not type(comp):
            self.errors.append(f"Type mismatch at {path}: Expected {type(base).__name__}, got {type(comp).__name__}")
            return

        if isinstance(base, dict):
            self._compare_dicts(base, comp, path)
        elif isinstance(base, list):
            self._compare_lists(base, comp, path)
        elif base != comp:
            self.errors.append(f"Value mismatch at {path}: Expected {base}, got {comp}")

    def _compare_dicts(self, base: Dict, comp: Dict, path: str):
        """
        Compare two dictionaries.
        """
        base_keys, comp_keys = set(base.keys()), set(comp.keys())

        # Detect missing and extra keys
        missing_keys = base_keys - comp_keys
        extra_keys = comp_keys - base_keys

        if missing_keys:
            self.errors.append(f"Missing keys at {path}: {sorted(missing_keys)}")
        if extra_keys:
            self.errors.append(f"Extra keys at {path}: {sorted(extra_keys)}")

        # Compare common keys
        for key in base_keys & comp_keys:
            self._compare_recursive(base[key], comp[key], f"{path}.{key}")

    def _compare_lists(self, base: List, comp: List, path: str):
        """
        Compare two lists.
        """
        if len(base) != len(comp):
            self.errors.append(f"List length mismatch at {path}: Expected {len(base)}, got {len(comp)}")
            return

        # Compare elements one by one
        for index, (item1, item2) in enumerate(zip(base, comp)):
            self._compare_recursive(item1, item2, f"{path}[{index}]")

In [3]:
if __name__ == "__main__":
    # Specify paths to JSON files
    base_file_path = "manual_run_silver_lakehouse_imagingstudy.json"
    target_file_path = "test.json"

    comparator = JSONComparator()
    discrepancies = comparator.compare_files(base_file_path, target_file_path)

    if discrepancies:
        print("Discrepancies found:")
        for error in discrepancies:
            print(f" - {error}")
    else:
        print("JSONs are identical.")

Discrepancies found:
 - File not found: [Errno 2] No such file or directory: 'manual_run_silver_lakehouse_imagingstudy.json'
